### Procesamiento de Lenguaje Natural I
**Desafío 1**

Alumno: Alexis Barniquez (a2203)



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cosa']

KeyError: 'cosa'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [ ]:
import pandas as pd

target_names = newsgroups_train.target_names

# Vectorización de documentos y Similitud

In [ ]:
# Se ajusta el vectorizador

X_train_tfidf = X_train

# Se seleccionan 5 documentos al azar
np.random.seed(42) # Fijamos semilla para reproducibilidad
random_indices = np.random.choice(X_train_tfidf.shape[0], 5, replace=False)

# Se calcula la similitud coseno y se imprimen los 5 más similares para cada uno
for idx in random_indices:
    # Similitud del documento seleccionado contra toda la matriz de entrenamiento
    sims = cosine_similarity(X_train_tfidf[idx], X_train_tfidf).flatten()

    # Se ordena de mayor a menor. Tomamos los top 6 (el 1ro será el propio documento)
    top_indices = sims.argsort()[-6:][::-1]

    print(f"\n{'='*80}")
    print(f"DOCUMENTO ORIGINAL [Clase Original: {target_names[y_train[idx]]}]")
    print(f"Extracto: {newsgroups_train.data[idx][:300]}...\n") # Se usa newsgroups_train.data para el texto original
    print("-" * 40)
    print("TOP 5 DOCUMENTOS MÁS SIMILARES:")

    for i, sim_idx in enumerate(top_indices[1:6], 1): # Omitimos el índice 0 (él mismo)
        print(f"\n{i}. [Clase: {target_names[y_train[sim_idx]]}] | Similitud: {sims[sim_idx]:.4f}")
        print(f"   Extracto: {newsgroups_train.data[sim_idx][:200]}...") # Se usa newsgroups_train.data para el texto original


DOCUMENTO ORIGINAL [Clase Original: comp.sys.mac.hardware]
Extracto: Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" ...

----------------------------------------
TOP 5 DOCUMENTOS MÁS SIMILARES:

1. [Clase: comp.sys.mac.hardware] | Similitud: 0.6665
   Extracto: Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyone out there who has found a really good deal on a Centris 650
send me the price.  I don't want to k...

2. [Clase: comp.sys.ibm.pc.hardware] | Similitud: 0.3476
   Extracto: Hay all:

    Has anyone out there heard of any performance stats on the fabled p24t.
 I was wondering what it's performance compared to the 486/66 and/or
pentium would be.  Any info would be helpful....

3. [Clase: comp.sys.mac

# Clasificador Zero-Shot (por prototipos)

In [ ]:
from sklearn.metrics import classification_report

# Transformamos el conjunto de test con el mismo vectorizador
X_test_tfidf = X_test

# Calculamos la similitud de cada documento de test con TODOS los de entrenamiento
# Resultado: Matriz de tamaño (N_test, N_train)
sims_test_train = cosine_similarity(X_test_tfidf, X_train_tfidf)

# Para cada documento de test, obtenemos el índice del doc de train más similar
best_train_indices = sims_test_train.argmax(axis=1)

# Asignamos la etiqueta original de ese documento de train al de test
y_pred_zero_shot = y_train[best_train_indices]

# ------------------- Reporte de Clasificación -----------------
print("Reporte de Clasificación - Modelo Basado en Similitud (Prototipos):\n")
print(classification_report(y_test, y_pred_zero_shot, target_names=target_names))

Reporte de Clasificación - Modelo Basado en Similitud (Prototipos):

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
      

# Optimización de Modelos Naïve Bayes

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Creamos un Pipeline que integra la vectorización y la clasificación
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),  # El ngram_range queda por defecto (1, 1)
    ('clf', MultinomialNB())       # Placeholder, se reemplazará en el GridSearch
])

# Espacio de búsqueda hiperparámetros
# RESTRICCIÓN: No se modifica ngram_range
parameters = [
    {
        'tfidf__max_df': [0.75, 0.90, 1.0], # Umbral máximo de frecuencia de documentos
        'tfidf__min_df': [1, 2, 5],         # Frecuencia mínima absoluta
        'clf': [MultinomialNB()],
        'clf__alpha': [0.01, 0.1, 1.0, 2.0] # Suavizado de Laplace/Lidstone
    },
    {
        'tfidf__max_df': [0.75, 0.90, 1.0],
        'tfidf__min_df': [1, 2, 5],
        'clf': [ComplementNB()], # Ideal para datasets desbalanceados
        'clf__alpha': [0.01, 0.1, 1.0, 2.0]
    }
]

# Buscamos maximizar f1_macro utilizando Validación Cruzada en Train
grid_search = GridSearchCV(
    pipeline,
    parameters,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1, # Usar todos los núcleos disponibles
    verbose=1
)

grid_search.fit(newsgroups_train.data, y_train)

# Extracción de resultados y evaluación en Test
best_model = grid_search.best_estimator_
print(f"\nMejores hiperparámetros encontrados:\n{grid_search.best_params_}")

y_pred_nb = best_model.predict(newsgroups_test.data)
f1_macro_test = f1_score(y_test, y_pred_nb, average='macro')
print(f"\nF1-Score Macro en el conjunto de TEST: {f1_macro_test:.4f}")

Fitting 3 folds for each of 72 candidates, totalling 216 fits

Mejores hiperparámetros encontrados:
{'clf': ComplementNB(), 'clf__alpha': 0.1, 'tfidf__max_df': 0.75, 'tfidf__min_df': 1}

F1-Score Macro en el conjunto de TEST: 0.6950


# Similitud de Palabras (Matriz Término-Documento)

In [ ]:
# Transponer la matriz documento-término
# (Obtenemos una matriz donde las filas son palabras y las columnas documentos)
X_term_doc = X_train_tfidf.T

# Definir las palabras
palabras_elegidas = ["cola", "new", "car", "file", "fire"]

# Obtenemos el diccionario del vocabulario {palabra: índice}
vocab = tfidfvect.vocabulary_
# Creamos el diccionario inverso {índice: palabra} para decodificar los resultados
reverse_vocab = {idx: word for word, idx in vocab.items()}

# ----------- Verificación y cálculo de similitud entre palabras ------------
print("\n--- BÚSQUEDA DE SIMILITUD LÉXICA ---")
for palabra in palabras_elegidas:
    if palabra in vocab:
        word_idx = vocab[palabra]

        # Similitud de esta palabra con TODAS las del vocabulario
        word_sims = cosine_similarity(X_term_doc[word_idx], X_term_doc).flatten()

        # Top 6 (excluyendo el índice 0 que es la palabra exacta)
        top_word_indices = word_sims.argsort()[-6:][::-1]

        print(f"\nPalabra objetivo: '{palabra}'")
        print("Las 5 palabras más similares:")
        for i, sim_idx in enumerate(top_word_indices[1:6], 1):
            sim_word = reverse_vocab[sim_idx]
            sim_score = word_sims[sim_idx]
            print(f"  {i}. {sim_word} (Score: {sim_score:.4f})")
    else:
        print(f"\n[ATENCIÓN] La palabra '{palabra}' NO existe en el vocabulario del corpus.")


--- BÚSQUEDA DE SIMILITUD LÉXICA ---

Palabra objetivo: 'cola'
Las 5 palabras más similares:
  1. rc4 (Score: 0.7934)
  2. pepsi (Score: 0.7354)
  3. coca (Score: 0.6034)
  4. rc2 (Score: 0.4943)
  5. rc (Score: 0.2591)

Palabra objetivo: 'new'
Las 5 palabras más similares:
  1. york (Score: 0.2756)
  2. the (Score: 0.2525)
  3. and (Score: 0.2330)
  4. to (Score: 0.2294)
  5. brand (Score: 0.2237)

Palabra objetivo: 'car'
Las 5 palabras más similares:
  1. cars (Score: 0.1797)
  2. criterium (Score: 0.1770)
  3. civic (Score: 0.1748)
  4. owner (Score: 0.1689)
  5. dealer (Score: 0.1681)

Palabra objetivo: 'file'
Las 5 palabras más similares:
  1. files (Score: 0.2151)
  2. swap (Score: 0.2119)
  3. 102nd (Score: 0.1959)
  4. edtorial (Score: 0.1952)
  5. 2304 (Score: 0.1952)

Palabra objetivo: 'fire'
Las 5 palabras más similares:
  1. kerosene (Score: 0.2355)
  2. compound (Score: 0.2219)
  3. started (Score: 0.2198)
  4. arraingement (Score: 0.2094)
  5. curiouser (Score: 0.2094)


¿Qué hace el código?
Transpone la matriz TF-IDF. Ahora, cada palabra está representada matemáticamente por un vector cuyas dimensiones son los documentos en los que aparece. Al calcular el coseno de estos vectores, estamos asumiendo la Hipótesis Distribucional: las palabras que aparecen en los mismos contextos (documentos) tienden a tener significados o temáticas similares.